# Full BRISC pipeline smoke test

`manifest → split → train ResNet18 + U-Net → save checkpoints → validation/test inference → validation calibration → frozen-threshold test fusion`


In [1]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from prepare_plane_split import build_aligned_manifest, build_experiment_split

SEED = 42
SPLIT_ROOT = PROJECT_ROOT / "data" / "experiments" / "pipeline_smoke_seed42"
TRAIN_OUTPUT_ROOT = PROJECT_ROOT / "results" / "pipeline_smoke" / "training"
PREDICTION_ROOT = PROJECT_ROOT / "results" / "pipeline_smoke" / "predictions"
CHECKPOINT_ROOT = PROJECT_ROOT / "checkpoints" / "pipeline_smoke"
REPORT_ROOT = PROJECT_ROOT / "results" / "pipeline_smoke" / "reports"
for directory in (SPLIT_ROOT, PREDICTION_ROOT, REPORT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

environment = os.environ.copy()
environment["PYTHONPATH"] = str(PROJECT_ROOT / "src")
environment["NO_ALBUMENTATIONS_UPDATE"] = "1"
environment["OMP_NUM_THREADS"] = "1"

def run_step(name: str, arguments: list[str]) -> None:
    print(f"\n--- {name} ---")
    completed = subprocess.run(
        [sys.executable, *arguments],
        cwd=PROJECT_ROOT,
        env=environment,
        text=True,
        capture_output=True,
    )
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()

print(f"Python: {sys.executable}")
print(f"Project: {PROJECT_ROOT}")

Python: /opt/anaconda3/envs/brainTumor/bin/python
Project: /Users/louinsleung/Documents/brainTumor


## 1.a small real-data split

In [2]:
manifest = pd.read_csv(PROJECT_ROOT / "brisc2025" / "manifest.csv")
aligned = build_aligned_manifest(manifest)
full_split = build_experiment_split(
    aligned,
    held_out_plane="none",
    seed=SEED,
    validation_fraction=0.2,
)

def cap_each_stratum(frame: pd.DataFrame, size: int, seed: int) -> pd.DataFrame:
    parts = []
    for _, group in frame.groupby(["tumor_label", "plane_label"]):
        parts.append(group.sample(n=min(size, len(group)), random_state=seed))
    return pd.concat(parts, ignore_index=True)

caps = {"train": 4, "validation": 2, "test": 2}
smoke_parts = []
for split, cap in caps.items():
    partition = full_split.loc[full_split["experiment_split"] == split]
    smoke_parts.append(cap_each_stratum(partition, cap, SEED))
smoke_split = pd.concat(smoke_parts, ignore_index=True)
split_file = SPLIT_ROOT / "split_manifest.csv"
smoke_split.to_csv(split_file, index=False)

counts = smoke_split.groupby(
    ["experiment_split", "tumor_label", "plane_label"]
).size().rename("images")
assert set(smoke_split["experiment_split"]) == {"train", "validation", "test"}
assert smoke_split.loc[
    smoke_split["tumor_label"] != "no_tumor", "mask_relative_path"
].notna().all()
print(smoke_split["experiment_split"].value_counts())
counts.to_frame()

experiment_split
train         48
validation    24
test          24
Name: count, dtype: int64


images
experiment_split tumor_label plane_label        
test             glioma      axial             2
                             coronal           2
                             sagittal          2
                 meningioma  axial             2
                             coronal           2
                             sagittal          2
                 no_tumor    axial             2
                             coronal           2
                             sagittal          2
                 pituitary   axial             2
                             coronal           2
                             sagittal          2
train            glioma      axial             4
                             coronal           4
                             sagittal          4
                 meningioma  axial             4
                             coronal           4
                             sagittal          4
                 no_tumor    axial             4
                             coronal           4
                             sagittal          4
                 pituitary   axial             4
                             coronal           4
                             sagittal          4
validation       glioma      axial             2
                             coronal           2
                             sagittal          2
                 meningioma  axial             2
                             coronal           2
                             sagittal          2
                 no_tumor    axial             2
                             coronal           2
                             sagittal          2
                 pituitary   axial             2
                             coronal           2
                             sagittal          2

## 2. Train models briefly

In [3]:
run_step(
    "Train ResNet18 classifier",
    [
        "src/train_classification.py",
        "--data-root", str(PROJECT_ROOT / "brisc2025"),
        "--split-root", str(SPLIT_ROOT),
        "--output-dir", str(TRAIN_OUTPUT_ROOT),
        "--checkpoint-dir", str(CHECKPOINT_ROOT),
        "--run-name", "classification",
        "--image-size", "64",
        "--batch-size", "8",
        "--head-epochs", "1",
        "--fine-tune-epochs", "1",
        "--num-workers", "0",
        "--device", "cpu",
        "--seed", str(SEED),
    ],
)


--- Train ResNet18 classifier ---


Run: classification
Device: cpu; train=48, validation=24
Epoch 01 [head]: train loss=1.6643, F1=0.2464; val loss=1.4006, F1=0.3571, accuracy=0.3750
Epoch 02 [fine_tune]: train loss=1.2644, F1=0.4037; val loss=1.2304, F1=0.4946, accuracy=0.4583
Best validation macro F1=0.4946 at epoch 2
Saved best checkpoint to /Users/louinsleung/Documents/brainTumor/checkpoints/pipeline_smoke/classification/best.pt



In [4]:
run_step(
    "Train U-Net segmenter",
    [
        "src/train_segmentation.py",
        "--data-root", str(PROJECT_ROOT / "brisc2025"),
        "--split-root", str(SPLIT_ROOT),
        "--output-dir", str(TRAIN_OUTPUT_ROOT),
        "--checkpoint-dir", str(CHECKPOINT_ROOT),
        "--run-name", "segmentation",
        "--image-size", "64",
        "--batch-size", "4",
        "--epochs", "2",
        "--num-workers", "0",
        "--device", "cpu",
        "--seed", str(SEED),
    ],
)

classification_checkpoint = CHECKPOINT_ROOT / "classification" / "best.pt"
segmentation_checkpoint = CHECKPOINT_ROOT / "segmentation" / "best.pt"
assert classification_checkpoint.exists()
assert segmentation_checkpoint.exists()
classification_checkpoint, segmentation_checkpoint


--- Train U-Net segmenter ---


Run: segmentation
Device: cpu; train=48, validation=24
Epoch 01: train loss=0.7678, positive Dice=0.0666; val loss=0.9565, positive Dice=0.0421, all-case Dice=0.0316
Epoch 02: train loss=0.6552, positive Dice=0.0598; val loss=0.6339, positive Dice=0.0000, all-case Dice=0.2083
Best validation positive Dice=0.0421 at epoch 1
Saved best checkpoint to /Users/louinsleung/Documents/brainTumor/checkpoints/pipeline_smoke/segmentation/best.pt



(PosixPath('/Users/louinsleung/Documents/brainTumor/checkpoints/pipeline_smoke/classification/best.pt'),
 PosixPath('/Users/louinsleung/Documents/brainTumor/checkpoints/pipeline_smoke/segmentation/best.pt'))

## 3. predict validation and test splits

In [5]:
prediction_runs = {}
for task, checkpoint, batch_size in (
    ("classification", classification_checkpoint, 8),
    ("segmentation", segmentation_checkpoint, 4),
):
    for split in ("validation", "test"):
        output_dir = PREDICTION_ROOT / f"{task}_{split}"
        run_step(
            f"Predict {task} {split}",
            [
                "src/predict.py",
                "--task", task,
                "--checkpoint", str(checkpoint),
                "--split", split,
                "--split-root", str(SPLIT_ROOT),
                "--data-root", str(PROJECT_ROOT / "brisc2025"),
                "--output-dir", str(output_dir),
                "--batch-size", str(batch_size),
                "--device", "cpu",
            ],
        )
        prediction_runs[(task, split)] = output_dir / "run.json"

assert all(path.exists() for path in prediction_runs.values())
prediction_runs


--- Predict classification validation ---


Saved 24 validation predictions to /Users/louinsleung/Documents/brainTumor/results/pipeline_smoke/predictions/classification_validation/run.json


--- Predict classification test ---


Saved 24 test predictions to /Users/louinsleung/Documents/brainTumor/results/pipeline_smoke/predictions/classification_test/run.json


--- Predict segmentation validation ---


Saved 24 validation predictions to /Users/louinsleung/Documents/brainTumor/results/pipeline_smoke/predictions/segmentation_validation/run.json


--- Predict segmentation test ---


Saved 24 test predictions to /Users/louinsleung/Documents/brainTumor/results/pipeline_smoke/predictions/segmentation_test/run.json



{('classification',
  'validation'): PosixPath('/Users/louinsleung/Documents/brainTumor/results/pipeline_smoke/predictions/classification_validation/run.json'),
 ('classification',
  'test'): PosixPath('/Users/louinsleung/Documents/brainTumor/results/pipeline_smoke/predictions/classification_test/run.json'),
 ('segmentation',
  'validation'): PosixPath('/Users/louinsleung/Documents/brainTumor/results/pipeline_smoke/predictions/segmentation_validation/run.json'),
 ('segmentation',
  'test'): PosixPath('/Users/louinsleung/Documents/brainTumor/results/pipeline_smoke/predictions/segmentation_test/run.json')}

## 4. Select thresholds using validation predictions only

In [6]:
threshold_file = REPORT_ROOT / "thresholds.json"
run_step(
    "Calibrate validation thresholds",
    [
        "src/calibrate_thresholds.py",
        "--classification-run", str(prediction_runs[("classification", "validation")]),
        "--segmentation-run", str(prediction_runs[("segmentation", "validation")]),
        "--output-file", str(threshold_file),
    ],
)
thresholds = json.loads(threshold_file.read_text())
thresholds


--- Calibrate validation thresholds ---


Selected classification threshold 0.730713 and area threshold 0.579834; saved /Users/louinsleung/Documents/brainTumor/results/pipeline_smoke/reports/thresholds.json



{'classification_threshold': 0.7307133078575134,
 'area_threshold': 0.579833984375,
 'classification_validation_balanced_accuracy': 0.8333333333333334,
 'segmentation_validation_balanced_accuracy': 0.6111111111111112,
 'validation_sample_count': 24,
 'tie_break': 'smallest_threshold'}

## 5. Freeze thresholds and fuse official-test predictions

In [7]:
fusion_file = REPORT_ROOT / "test_fusion.json"
run_step(
    "Fuse test predictions",
    [
        "src/decision_fusion.py",
        "--classification-run", str(prediction_runs[("classification", "test")]),
        "--segmentation-run", str(prediction_runs[("segmentation", "test")]),
        "--classification-threshold", str(thresholds["classification_threshold"]),
        "--area-threshold", str(thresholds["area_threshold"]),
        "--output-file", str(fusion_file),
    ],
)
fusion_report = json.loads(fusion_file.read_text())
summary = fusion_report["summary"]
assert summary["sample_count"] == len(
    smoke_split.loc[smoke_split["experiment_split"] == "test"]
)
assert summary["classification_threshold"] == thresholds["classification_threshold"]
assert summary["area_threshold"] == thresholds["area_threshold"]
pd.Series(summary, name="full_pipeline_smoke_result")


--- Fuse test predictions ---


Saved 24 decisions and 13 inconsistencies to /Users/louinsleung/Documents/brainTumor/results/pipeline_smoke/reports/test_fusion.json



classification_threshold                            0.730713
area_threshold                                      0.579834
sample_count                                       24.000000
consistent_count                                   11.000000
inconsistent_count                                 13.000000
agreement_rate                                      0.458333
uncertain_rate                                      0.541667
type_a_count                                        7.000000
type_b_count                                        6.000000
classification_class_error_rate_consistent          0.545455
classification_presence_error_rate_consistent       0.090909
segmentation_presence_error_rate_consistent         0.090909
classification_class_error_rate_inconsistent        0.384615
classification_presence_error_rate_inconsistent     0.307692
segmentation_presence_error_rate_inconsistent       0.692308
Name: full_pipeline_smoke_result, dtype: float64